# 10 - Exception Handling, Memory Management & Performance

Concise, interview/revision focused. Part of Python & DSA.

# Part 1 - Exception Handling Deep Dive

### try/except/else/finally, and custom exception hierarchies

`else` runs only if no exception was raised; `finally` always runs. Real applications define a small exception hierarchy so callers can catch broadly (`AppError`) or specifically (`ValidationError`), instead of catching bare `Exception` everywhere.

In [1]:
class AppError(Exception):
    """Base for all errors this app raises on purpose."""

class ValidationError(AppError):
    def __init__(self, field, reason):
        self.field, self.reason = field, reason
        super().__init__(f"{field}: {reason}")

class NotFoundError(AppError):
    pass

def get_user(user_id):
    if user_id < 0:
        raise ValidationError("user_id", "must be non-negative")
    if user_id > 100:
        raise NotFoundError(f"user {user_id} does not exist")
    return {"id": user_id}

for uid in [5, -1, 999]:
    try:
        print(get_user(uid))
    except ValidationError as e:
        print("bad input:", e)
    except NotFoundError as e:
        print("not found:", e)
    except AppError as e:                 # catches any future subclass too
        print("other app error:", e)
    else:
        print("no error")
    finally:
        print("  (checked user", uid, ")")

{'id': 5}
no error
  (checked user 5 )
bad input: user_id: must be non-negative
  (checked user -1 )
not found: user 999 does not exist
  (checked user 999 )


### Exception chaining: raise ... from ...

Preserves the ORIGINAL traceback as `__cause__` when re-raising a different exception type -- critical for debugging; without it, the root cause is lost.

In [2]:
def load_config():
    try:
        int("not-a-number")
    except ValueError as e:
        raise AppError("config file is corrupted") from e   # keeps the original ValueError visible

try:
    load_config()
except AppError as e:
    print("raised:", e)
    print("caused by:", repr(e.__cause__))

raised: config file is corrupted
caused by: ValueError("invalid literal for int() with base 10: 'not-a-number'")


### except* (3.11+): handling multiple independent errors from one block

Useful with `asyncio.gather` / concurrent code, where several tasks can fail independently and you want to handle each error type without one failure hiding the rest.

In [3]:
try:
    raise ExceptionGroup("multiple failures", [ValueError("bad value"), TypeError("bad type")])
except* ValueError as eg:
    print("caught ValueErrors:", eg.exceptions)
except* TypeError as eg:
    print("caught TypeErrors:", eg.exceptions)

caught ValueErrors: (ValueError('bad value'),)
caught TypeErrors: (TypeError('bad type'),)


# Part 2 - Memory Management

### Reference counting

CPython frees an object the moment its reference count hits zero -- this is why simple cases are collected instantly, no garbage collector pause needed.

In [4]:
import sys

a = [1, 2, 3]
print("refcount via a:", sys.getrefcount(a) - 1)   # -1: getrefcount's own temporary argument reference

b = a
print("after b = a:", sys.getrefcount(a) - 1)

del b
print("after del b:", sys.getrefcount(a) - 1)

refcount via a: 1
after b = a: 2
after del b: 1


### Circular references and the garbage collector

Reference counting alone cannot free a cycle (two objects referencing each other) even when nothing external points to either -- the generational `gc` module exists specifically to find and collect these.

In [5]:
import gc

class Node:
    def __init__(self, name):
        self.name = name
        self.other = None
    def __del__(self):
        print(f"Node({self.name}) collected")

def make_cycle():
    n1, n2 = Node("A"), Node("B")
    n1.other = n2
    n2.other = n1          # a cycle: A -> B -> A
    return None            # both go out of scope here, but refcount never hits 0 alone

gc.disable()
make_cycle()
print("with gc disabled: nothing printed above -- the cycle leaked")
gc.collect()
print("after manual gc.collect(): both nodes freed")
gc.enable()

with gc disabled: nothing printed above -- the cycle leaked
Node(A) collected
Node(B) collected
after manual gc.collect(): both nodes freed


### weakref: referencing without keeping alive

A `weakref` does not increase the refcount, so it does not, by itself, keep an object alive or create a cycle -- common for caches and observer patterns where you want to reference an object without owning it.

In [6]:
import weakref

class Cache:
    pass

obj = Cache()
ref = weakref.ref(obj)
print("alive:", ref() is not None)

del obj
print("after del:", ref() is not None)   # gone -- the weakref did not keep it alive

alive: True
after del: False


### __slots__: trading flexibility for memory

By default every instance carries a `__dict__` for arbitrary attributes. `__slots__` declares a fixed attribute set up front and skips that `__dict__` entirely -- meaningful savings when creating millions of small objects (e.g. graph nodes, feature vectors).

In [7]:
import sys

class PointDict:
    def __init__(self, x, y):
        self.x, self.y = x, y

class PointSlots:
    __slots__ = ("x", "y")
    def __init__(self, x, y):
        self.x, self.y = x, y

d, s = PointDict(1, 2), PointSlots(1, 2)
print("with __dict__:   ", sys.getsizeof(d) + sys.getsizeof(d.__dict__), "bytes (obj + its __dict__)")
print("with __slots__:  ", sys.getsizeof(s), "bytes")

try:
    s.z = 3   # slots also prevents ad-hoc new attributes -- catches typos as errors, not silent bugs
except AttributeError as e:
    print("blocked:", e)

with __dict__:    344 bytes (obj + its __dict__)
with __slots__:   48 bytes
blocked: 'PointSlots' object has no attribute 'z'


# Part 3 - Profiling & Performance

### cProfile: where is the time actually going

Guessing where a function is slow is unreliable; `cProfile` gives per-function call counts and cumulative time. Always profile before optimizing.

In [8]:
import cProfile, pstats, io

def slow_fn():
    total = 0
    for i in range(200_000):
        total += i ** 2
    return total

def caller():
    return [slow_fn() for _ in range(3)]

profiler = cProfile.Profile()
profiler.enable()
caller()
profiler.disable()

stream = io.StringIO()
stats = pstats.Stats(profiler, stream=stream).sort_stats("tottime")
stats.print_stats("slow_fn|caller")   # filter to our own functions -- ignores IPython's wrapper frames
print(stream.getvalue())

         46 function calls in 0.064 seconds

   Ordered by: internal time
   List reduced from 21 to 2 due to restriction <'slow_fn|caller'>

   ncalls  tottime  percall  cumtime  percall filename:lineno(function)
        3    0.064    0.021    0.064    0.021 /tmp/ipykernel_596/3772571208.py:3(slow_fn)
        1    0.000    0.000    0.064    0.064 /tmp/ipykernel_596/3772571208.py:9(caller)





### tracemalloc: where memory is actually going

Standard library, no install needed. Snapshots the allocator to show which lines hold the most memory -- the memory equivalent of cProfile.

In [9]:
import tracemalloc

tracemalloc.start()
data = [str(i) * 10 for i in range(50_000)]     # deliberately wasteful allocation
snapshot = tracemalloc.take_snapshot()
top = snapshot.statistics("lineno")[0]
print(top)
tracemalloc.stop()

/tmp/ipykernel_596/4261646280.py:4: size=4769 KiB, count=50001, average=98 B


### A classic optimization that modern CPython has largely made obsolete

Binding a global/builtin to a local name before a hot loop (`_len = len`) used to give a measurable speedup, since local variable access was faster than a global lookup. Since Python 3.11's specializing adaptive interpreter, this gap has mostly closed for simple cases -- worth measuring below rather than assuming, since "optimization folklore" like this drifts out of date as the interpreter improves.

In [10]:
import time

def with_repeated_lookup(n):
    total = 0
    for i in range(n):
        total += len(str(i))          # looks up `len` and `str` freshly every iteration
    return total

def with_local_binding(n):
    _len, _str = len, str             # bind once, outside the loop
    total = 0
    for i in range(n):
        total += _len(_str(i))
    return total

n = 2_000_000
t1 = time.perf_counter(); with_repeated_lookup(n); t1 = time.perf_counter() - t1
t2 = time.perf_counter(); with_local_binding(n); t2 = time.perf_counter() - t2
print(f"repeated global lookups: {t1:.3f}s")
print(f"local-bound:             {t2:.3f}s")
print("on Python 3.11+, expect these to be close -- measure before trusting old advice like this")

repeated global lookups: 0.313s
local-bound:             0.290s
on Python 3.11+, expect these to be close -- measure before trusting old advice like this


## Interview rapid-fire

- Python has BOTH refcounting (immediate, deterministic) and a cyclic GC (periodic, for cycles refcounting cannot resolve) -- most objects never touch the cyclic GC at all.
- `__del__` is not guaranteed to run at a predictable time, and never for objects alive when the interpreter exits -- do not rely on it for critical cleanup; use a context manager instead.
- `__slots__` disables `__dict__` (and multiple inheritance from other slotted classes with different slots), so it is a real memory/flexibility tradeoff, not a free win.
- `raise X from Y` vs a bare `raise X` inside an except block: the bare form still sets `__context__` automatically, but `from` sets the more informative `__cause__` and marks it explicit in the traceback.

## Practice

1. Write `retryable_error_hierarchy`: a `TransientError` and `PermanentError` both subclassing `ServiceError`, then a function that retries only on `TransientError`.
2. Profile a function that builds a list with `.append()` in a loop vs one built with a list comprehension -- confirm which is faster with `cProfile` or `timeit`.
3. Convert a class with 5 attributes to use `__slots__` and measure the `sys.getsizeof` difference for 100,000 instances.